# Leaf boundary layer

> 

In [ ]:
# | default_exp leaf_boundary_layer

In [ ]:
# | hide
from fastcore import *
from nbdev.showdoc import *

In [ ]:
# | export
from plant_hydraulics.parameter_classes import PhysCon, Atmos, Leaf, Flux

In [ ]:
# | export


def leaf_boundary_layer(physcon: PhysCon, atmos: Atmos, leaf: Leaf, flux: Flux) -> Flux:
    """
    Calculate leaf boundary layer conductances.
    
    Leaf physiology 101:
    
    - The boundary layer: Thin layer that regulates the transfer of heat, water 
    vapor and CO2 between the leaf surface and the atmosphere. Thinner boundary 
    layers associated with small leaves have on avarage higher conductances. 
    
    Keep in mind that dessert plants might cool themselves by sensible heat. 
    Small leaves produce thin boundary layers, which causes high conductance of 
    heat thorught the boundary layer (high gbh) and high heat conduction. 
    Therefore it could be said that sensible heat and not transpiration cool the 
    leaves in desserts.    

    __Parameters:__

    - PhysCon: Physical constants

        - grav: float
            Gravitational acceleration (m/s2).
        - tfrz: float
            Freezing point of water (K).
        - visc0: float
            Kinematic viscosity at 0°C and 1013.25 hPa (m2/s).
        - Dh0: float
            Molecular diffusivity (heat) at 0°C and 1013.25 hPa (m2/s).
        - Dv0: float
            Molecular diffusivity (H2O) at 0°C and 1013.25 hPa (m2/s).
        - Dc0: float
            Molecular diffusivity (CO2) at 0°C and 1013.25 hPa (m2/s).

    - Atmos: Atmospheric forcing variables

        - patm: Atmospheric pressure (Pa).
        - rhomol: Molar density (mol/m3).
        - wind: Wind speed (m/s).
        - tair: Air temperature (K).

    - Leaf: Leaf Object

        - dleaf: Leaf dimension (m).

    - Flux: Flux variables with the following input

        - tleaf: Leaf temperature (K).

    __Returns:__

    - Flux: Updated flux object with the following attributes:

        - gbh: Leaf boundary layer conductance for heat (mol/m2 leaf/s).
        - gbv: Leaf boundary layer conductance for H2O (mol H2O/m2 leaf/s).
        - gbc: Leaf boundary layer conductance for CO2 (mol CO2/m2 leaf/s).


    Parameters
    ----------
    physcon : PhysCon
        Physical constants:
        - grav : float
            Gravitational acceleration (m/s2).
        - tfrz : float
            Freezing point of water (K).
        - visc0 : float
            Kinematic viscosity at 0°C and 1013.25 hPa (m2/s).
        - Dh0 : float
            Molecular diffusivity (heat) at 0°C and 1013.25 hPa (m2/s).
        - Dv0 : float
            Molecular diffusivity (H2O) at 0°C and 1013.25 hPa (m2/s).
        - Dc0 : float
            Molecular diffusivity (CO2) at 0°C and 1013.25 hPa (m2/s).
    atmos : Atmos
        Atmospheric forcing variables:
        - patm : float
            Atmospheric pressure (Pa).
        - rhomol : float
            Molar density (mol/m3).
        - wind : float
            Wind speed (m/s).
        - tair : float
            Air temperature (K).
    leaf : Leaf
        Leaf parameters:
        - dleaf : float
            Leaf dimension (m).
    flux : Flux
        Flux variables with the following input:
        - tleaf : float
            Leaf temperature (K).

    Returns
    -------
    flux : Flux
        Updated flux object with the following attributes:
        - gbh : float
            Leaf boundary layer conductance for heat (mol/m2 leaf/s).
        - gbv : float
            Leaf boundary layer conductance for H2O (mol H2O/m2 leaf/s).
        - gbc : float
            Leaf boundary layer conductance for CO2 (mol CO2/m2 leaf/s).
    """

    # Constants -----------------------------------------------------------------
    
    # Adjust diffusivity for temperature and pressure
    fac = 101325.0 / atmos.patm * (atmos.tair / physcon.tfrz) ** 1.81

    # Kinematic viscosity (m2/s)
    visc = physcon.visc0 * fac

    # Molecular diffusivity, heat (m2/s)
    Dh = physcon.Dh0 * fac

    # Molecular diffusivity, H2O (m2/s)
    Dv = physcon.Dv0 * fac

    # Molecular diffusivity, CO2 (m2/s)
    Dc = physcon.Dc0 * fac

    # Empirical correction factor. Necessary because leaves are not flat and 
    # rectangular plates
    
    b1 = 1.5
    
    # Dimensionless numbers -----------------------------------------------------

    # Reynolds number
    Re = atmos.wind * leaf.dleaf / visc

    # Prandtl number
    Pr = visc / Dh

    # Schmidt number for H2O
    Scv = visc / Dv

    # Schmidt number for CO2
    Scc = visc / Dc

    # Grashof number
    Gr = (
        physcon.grav
        * leaf.dleaf**3
        * max(flux.tleaf - atmos.tair, 0.0)
        / (atmos.tair * visc * visc)
    )

    # Nusselt number (Nu) and Sherwood numbers (H2O: Shv, CO2: Shc) -------------

    # Forced convection
    # Eq 10.33
    Nu_lam = b1 * 0.66 * Pr ** (0.33) * Re ** (0.5)
    Shv_lam = b1 * 0.66 * Scv ** (0.33) * Re ** (0.5)
    Shc_lam = b1 * 0.66 * Scc ** (0.33) * Re ** (0.5)

    # Forced convection - turbulent flow ----------------------------------------
    Nu_turb = b1 * 0.036 * Pr ** (0.33) * Re ** (0.8)
    Shv_turb = b1 * 0.036 * Scv ** (0.33) * Re ** (0.8)
    Shc_turb = b1 * 0.036 * Scc ** (0.33) * Re ** (0.8)

    # Choose correct flow regime for forced convection --------------------------
    Nu_forced = max(Nu_lam, Nu_turb)
    Shv_forced = max(Shv_lam, Shv_turb)
    Shc_forced = max(Shc_lam, Shc_turb)

    # Free convection -----------------------------------------------------------
    Nu_free = 0.54 * Pr ** (0.25) * Gr ** (0.25)
    Shv_free = 0.54 * Scv ** (0.25) * Gr ** (0.25)
    Shc_free = 0.54 * Scc ** (0.25) * Gr ** (0.25)

    # Combined forced and free convection ---------------------------------------
    Nu = Nu_forced + Nu_free
    Shv = Shv_forced + Shv_free
    Shc = Shc_forced + Shc_free

    # Boundary layer conductances (m/s) -----------------------------------------
    
    # Heat
    flux.gbh = Dh * Nu / leaf.dleaf
    
    # Water vapour
    flux.gbv = Dv * Shv / leaf.dleaf
    
    # CO2
    flux.gbc = Dc * Shc / leaf.dleaf

    # Convert conductance (m/s) to (mol/m2/s) -----------------------------------

    # How easily heat passes through (controls whether the leaf can cool itself)
    flux.gbh *= atmos.rhomol

    # How easily water vapor passes through (controls transpiration)
    flux.gbv *= atmos.rhomol

    # How easily CO2 passes through (controls photosynthesis)
    flux.gbc *= atmos.rhomol

    return flux